# Task 5 · Colour reliability and geometric recovery
Use a CPU runtime with the Task 3 candidate cache, dataset and a full Task 4
training run in Drive. The notebook verifies or produces the auxiliary
surface metrics and masks, changes RGB while holding XYZ fixed, fits the
plane refinements, and measures the effect of paint on flat mesh patches.
Set TASK4_RUN to select an exact run for a paired comparison.


In [ ]:
from pathlib import Path
from google.colab import drive
import json
import os
import subprocess
import sys

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/MVP_Project')
TASK4_RUN = ''  # Set to a full_* run folder name. The default uses the latest complete run.
data = PROJECT / 'task3/gso_task3_v1'
runs = PROJECT / 'task4'

def complete(path):
    required = ('run.json', 'per_object.csv', 'geometry.pt', 'plain_rgb.pt', 'gated_rgb.pt')
    return all((path / filename).is_file() for filename in required)

if TASK4_RUN:
    run = runs / TASK4_RUN
    if not complete(run):
        raise FileNotFoundError(f'Incomplete Task 4 run: {run}')
else:
    full_runs = sorted((p for p in runs.glob('full_*') if complete(p)), reverse=True)
    if not full_runs:
        raise FileNotFoundError(f'Run the Task 4 training notebook first. No complete runs in {runs}')
    run = full_runs[0]

manifest = json.loads((data / 'manifest.json').read_text())
candidates = data.parent / f"candidates_{manifest['revision'][:8]}_{manifest['points']}_{manifest['seed']}_v1"
pgd_cache = runs / 'pgd_cache'
for required in (candidates, pgd_cache):
    if not required.is_dir():
        raise FileNotFoundError(required)
print('Evaluating:', run)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trimesh==5.2.2', 'point-cloud-utils'], check=True)


In [ ]:
sources = {'data/ranking.py': 'import numpy as np\nimport pandas as pd\nfrom sklearn.neighbors import NearestNeighbors\n\nDATASET_COMPOSITION = (20, 10, 10, 20) # agreement, disagreement (geometry varies), disagreement (texture varies), random samples\n\ndef compute_scores(xyz, rgb, k=24):\n    # Find neighbours in clean 3D geometry.\n    knn = NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree").fit(xyz)\n    indices = knn.kneighbors(xyz, return_distance=False)[:, 1:]\n    neighbours = xyz[indices]  # (N, k, 3)\n\n    # Local geometric variation via PCA.\n    centered = neighbours - neighbours.mean(axis=1, keepdims=True)\n    covariance = (np.einsum("nki,nkj->nij", centered, centered) / k)\n    eigenvalues = np.linalg.eigvalsh(covariance)\n    geometry_score = (eigenvalues[:, 0] / (eigenvalues.sum(axis=1) + 1e-12))\n\n    # Local RGB variation.\n    color_distances = np.linalg.norm(rgb[indices] - rgb[:, None, :], axis=2)\n    color_score = color_distances.mean(axis=1)\n\n    return geometry_score, color_score\n\ndef compute_thresholds(objects):\n    """\n    objects: dict mapping object_id -> (geometry_scores, colour_scores)\n    """\n    all_g = np.concatenate([g for g, c in objects.values()])\n    all_c = np.concatenate([c for g, c in objects.values()])\n\n    return {\n        "g_high": np.quantile(all_g, 0.90),\n        "g_low": np.quantile(all_g, 0.50),\n        "c_high": np.quantile(all_c, 0.90),\n        "c_low": np.quantile(all_c, 0.50),\n    }\n\n\ndef summarize_object(g, c, thresholds):\n    gh = thresholds["g_high"]\n    gl = thresholds["g_low"]\n    ch = thresholds["c_high"]\n    cl = thresholds["c_low"]\n\n    texture_only = np.mean((c >= ch) & (g <= gl))\n    geometry_only = np.mean((g >= gh) & (c <= cl))\n    aligned = np.mean((g >= gh) & (c >= ch))\n\n    return {\n        "texture_only": float(texture_only),\n        "geometry_only": float(geometry_only),\n        "aligned": float(aligned),\n        "disagreement": float(texture_only + geometry_only),\n    }\n\ndef select_objects(ranking, seed=2025, composition=DATASET_COMPOSITION):\n    selected = set()\n    groups = {}\n\n    agreement, geometry_only, texture_only, random_samples = composition\n\n    criteria = [\n        ("agreement", "aligned", agreement),\n        ("texture_only", "texture_only", texture_only),\n        ("geometry_only", "geometry_only", geometry_only),\n    ]\n\n    for group, column, count in criteria:\n        ranked = ranking.sort_values([column, "object_id"], ascending=[False, True])\n        choices = [obj for obj in ranked["object_id"] if obj not in selected][:count]\n\n        if len(choices) != count:\n            raise ValueError(f"Not enough objects for {group}")\n\n        groups[group] = choices\n        selected.update(choices)\n\n    remaining = ranking.loc[~ranking["object_id"].isin(selected), "object_id", ].to_numpy()\n    rng = np.random.default_rng(seed)\n    groups["random"] = list(rng.choice(remaining, size=random_samples, replace=False))\n\n    return groups\n\ndef create_splits(groups, seed=2025):\n    rng = np.random.default_rng(seed)\n    assignments = {}\n\n    for group, ids in groups.items():\n        ids = list(ids)\n        rng.shuffle(ids)\n\n        n_train = int(len(ids) * 0.7)\n        n_val = int(len(ids) * 0.1)\n\n        for i, object_id in enumerate(ids):\n            split = (\n                "train" if i < n_train\n                else "val" if i < n_train + n_val\n                else "test"\n            )\n\n            assignments[object_id] = {"split": split, "group": group}\n\n    return assignments\n\ndef rank_candidates(candidates):\n    objects = {}\n\n    for candidate in candidates:\n        points, colors, meta = candidate\n        object_id = str(meta["object_id"])\n        objects[object_id] = compute_scores(points, colors)\n\n    thresholds = compute_thresholds(objects)\n\n    rows = [\n            {"object_id": object_id,\n             **summarize_object(g, c, thresholds)}\n            for object_id, (g, c) in objects.items()\n        ]\n    \n    ranking = pd.DataFrame(rows)\n    return ranking', 'task4/train_rgb_pgd.py': '"""Train matched geometry, RGB and gated RGB residual heads on frozen PGD."""\nimport argparse\nimport csv\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nMODES = ("geometry", "plain_rgb", "gated_rgb")\nXYZ_LEVELS = (0.01, 0.02, 0.025)\nRGB_LEVELS = (0.0, 0.05)\nPATCH_SIZE = 1000\nPATCH_OVERSAMPLING = 6\nNEIGHBORS = 16\n\n\ndef mlp(inputs, outputs):\n    return nn.Sequential(nn.Linear(inputs, 64), nn.ReLU(),\n                         nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, outputs))\n\n\nclass ResidualHead(nn.Module):\n    """Same trainable layers in all modes. Only the inputs and gate pooling differ."""\n\n    def __init__(self, mode):\n        super().__init__()\n        if mode not in MODES:\n            raise ValueError(f"Unknown model: {mode}")\n        self.mode = mode\n        self.geometry = mlp(9, 3)\n        self.fusion = mlp(15, 4)\n        for head in (self.geometry, self.fusion):\n            nn.init.zeros_(head[-1].weight)\n            nn.init.zeros_(head[-1].bias)\n\n    def forward(self, base, noisy, indices, rgb):\n        relative = base[indices] - base[:, None]\n        local_geometry = torch.cat(\n            [relative.mean(dim=1), relative.abs().mean(dim=1)], dim=-1\n        )\n        geometry = torch.cat([base - noisy, local_geometry], dim=-1)\n        if self.mode == "geometry":\n            context = local_geometry\n        else:\n            relative_rgb = rgb[indices] - rgb[:, None]\n            context = torch.cat(\n                [relative_rgb.mean(dim=1), relative_rgb.abs().mean(dim=1)], dim=-1\n            )\n\n        mixed = self.fusion(torch.cat([geometry, context], dim=-1))\n        reference = self.fusion(torch.cat([geometry, torch.zeros_like(context)], dim=-1))\n        # Isolate the change caused by context. Uniform RGB gives zero colour correction.\n        contextual_delta = mixed[:, :3] - reference[:, :3]\n        logits = mixed[:, 3:]\n        if self.mode != "gated_rgb":\n            logits = logits.mean(dim=0, keepdim=True).expand_as(logits)\n        gain = logits.sigmoid()\n        return base + self.geometry(geometry) + gain * contextual_delta, gain\n\n\n@dataclass\nclass Sample:\n    oid: str\n    split: str\n    group: str\n    noise: float\n    xyz: np.ndarray\n    clean: np.ndarray\n    colors: dict\n    metric_scale: float\n\n\ndef load_cloud(path):\n    with np.load(path, allow_pickle=False) as arrays:\n        xyz, rgb = (arrays[key].astype(np.float32) for key in ("xyz", "rgb"))\n    if (xyz.shape != rgb.shape or xyz.ndim != 2 or xyz.shape[1] != 3 or\n            not np.isfinite(xyz).all() or not np.isfinite(rgb).all() or\n            np.any(rgb < 0) or np.any(rgb > 1)):\n        raise ValueError(f"Invalid XYZRGB arrays in {path}")\n    return xyz, rgb\n\n\ndef load_samples(root, smoke):\n    assignments = json.loads((root / "selection.json").read_text())\n    split_ids = {split: sorted(oid for oid, info in assignments.items() if info["split"] == split)\n                 for split in ("train", "val", "test")}\n    if set().union(*map(set, split_ids.values())) != set(assignments):\n        raise ValueError("Unexpected split label in selection.json")\n    samples = {}\n    for split, names in split_ids.items():\n        minimum = 2 if split == "train" else 1\n        if len(names) < minimum:\n            raise ValueError(f"{split} requires at least {minimum} distinct objects")\n        samples[split] = []\n        for oid in names[:minimum] if smoke else names:\n            if Path(oid).name != oid or oid in {".", ".."}:\n                raise ValueError(f"Invalid object ID: {oid}")\n            folder = root / split / oid\n            clean, _ = load_cloud(folder / "clean.npz")\n            clean_center = (clean.max(0) + clean.min(0)) / 2\n            clean_radius = np.linalg.norm(clean - clean_center, axis=1).max()\n            if len(clean) < PATCH_SIZE or clean_radius <= 0:\n                raise ValueError(f"Invalid clean point cloud for {oid}")\n            for noise in (0.02,) if smoke else XYZ_LEVELS:\n                xyz, original = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                other_xyz, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                if xyz.shape != clean.shape or not np.array_equal(xyz, other_xyz):\n                    raise ValueError(f"Point count or paired geometry mismatch for {oid} at {noise}")\n                center = (xyz.max(0) + xyz.min(0)) / 2\n                radius = np.linalg.norm(xyz - center, axis=1).max()\n                if radius <= 0:\n                    raise ValueError(f"Degenerate input cloud for {oid}")\n                samples[split].append(Sample(\n                    oid, split, assignments[oid]["group"], noise,\n                    (xyz - center) / radius, (clean - center) / radius,\n                    {"original": original, "jitter": jitter}, float(radius / clean_radius),\n                ))\n    return samples\n\n\ndef pgd_patches(model, points, knn, fps, iterations):\n    """Stitch predictions at their original point indices so each RGB stays aligned."""\n    center_count = max(1, PATCH_OVERSAMPLING * len(points) // PATCH_SIZE)\n    best = points.new_full((len(points),), float("inf"))\n    for _ in range(iterations):\n        centers, _ = fps(points[None], center_count)\n        result = torch.empty_like(points)\n        best.fill_(float("inf"))\n\n        def apply_patches(patch_centers):\n            distances, indices, neighborhoods = knn(\n                patch_centers[None], points[None], K=PATCH_SIZE, return_nn=True,\n            )\n            for center, dists, ids, patch in zip(\n                patch_centers, distances[0], indices[0], neighborhoods[0]\n            ):\n                relative = dists / dists[-1].clamp_min(1e-12)\n                predicted = model.denoise_langevin_dynamics((patch - center)[None])[0] + center\n                take = relative < best[ids]\n                result[ids[take]] = predicted[take]\n                best[ids[take]] = relative[take]\n\n        apply_patches(centers[0])\n        # FPS patches can miss a few rows. Process centred patches for any such rows.\n        for idx in torch.where(~best.isfinite())[0].tolist():\n            if not torch.isfinite(best[idx]):\n                apply_patches(points[idx:idx + 1])\n        if not best.isfinite().all() or not result.isfinite().all():\n            raise RuntimeError("PGD stitching failed to reconstruct every input row")\n        points = result\n    return points\n\n\ndef digest_file(path):\n    with path.open("rb") as stream:\n        sha = hashlib.sha256()\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef precompute(samples, pgd_root, checkpoint, iterations, cache, seed, device, knn):\n    sys.path.insert(0, str(pgd_root))\n    from models.pgd import PGDModel\n    from models.utils import farthest_point_sampling\n\n    revision = subprocess.check_output(["git", "-C", str(pgd_root), "rev-parse", "HEAD"], text=True).strip()\n    pgd_config = dict(revision=revision, weights=digest_file(checkpoint), iterations=iterations,\n                      patch_size=PATCH_SIZE, oversampling=PATCH_OVERSAMPLING, seed=seed,\n                      stitch="indexed_v1", device=torch.cuda.get_device_name(device),\n                      torch=torch.__version__)\n    model = None\n    cache.mkdir(parents=True, exist_ok=True)\n    prepared = {}\n    started = time.monotonic()\n    all_samples = [sample for split in samples.values() for sample in split]\n    for number, sample in enumerate(all_samples, 1):\n        key = hashlib.sha256(json.dumps(pgd_config, sort_keys=True).encode() + sample.xyz.tobytes())\n        filename = cache / f"{key.hexdigest()}.npy"\n        if filename.is_file():\n            baseline = np.load(filename, allow_pickle=False)\n        else:\n            if model is None:\n                model = PGDModel.load_from_checkpoint(str(checkpoint), map_location="cpu")\n                model = model.to(device).requires_grad_(False).eval()\n            # Stable result for each cache entry independently of processing order.\n            torch.manual_seed(seed)\n            with torch.no_grad():\n                baseline = pgd_patches(model, torch.from_numpy(sample.xyz).to(device), knn,\n                                       farthest_point_sampling, iterations).cpu().numpy()\n            partial = filename.with_suffix(".part")\n            with partial.open("wb") as stream:\n                np.save(stream, baseline)\n            partial.replace(filename)\n        if baseline.shape != sample.xyz.shape or not np.isfinite(baseline).all():\n            raise ValueError(f"Invalid frozen prediction for {sample.oid}")\n        xyz = torch.from_numpy(sample.xyz.copy()).to(device)\n        with torch.no_grad():\n            ids = knn(xyz[None], xyz[None], K=NEIGHBORS + 1).idx[0]\n            self_index = torch.arange(len(xyz), device=device)[:, None]\n            order = (ids == self_index).to(torch.int32).argsort(dim=1, stable=True)\n            neighbors = ids.gather(1, order[:, :NEIGHBORS]).cpu()\n        prepared[(sample.oid, sample.noise)] = (torch.from_numpy(baseline), neighbors)\n        print(f"PGD {number}/{len(all_samples)} · {sample.split}/{sample.oid} · {sample.noise:g}", flush=True)\n    print(f"PGD preparation took {time.monotonic() - started:.1f} s", flush=True)\n    return prepared, pgd_config\n\n\ndef batch(sample, prepared, condition, device):\n    base, neighbors = prepared[(sample.oid, sample.noise)]\n    return (base.to(device), torch.from_numpy(sample.xyz).to(device),\n            neighbors.to(device), torch.from_numpy(sample.colors[condition]).to(device),\n            torch.from_numpy(sample.clean).to(device))\n\n\ndef chamfer(prediction, target, knn):\n    a = knn(prediction[None], target[None], K=1).dists\n    b = knn(target[None], prediction[None], K=1).dists\n    return a.mean() + b.mean()\n\n\ndef fit(mode, samples, prepared, epochs, seed, device, knn, directory):\n    torch.manual_seed(seed)\n    model = ResidualHead(mode).to(device)\n    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    rng = np.random.default_rng(seed)\n    color_conditions = ("original", "jitter")\n    best_score, best_state = math.inf, None\n    history = []\n    for epoch in range(epochs):\n        started = time.monotonic()\n        model.train()\n        train_losses = []\n        for index in rng.permutation(len(samples["train"])):\n            sample = samples["train"][index]\n            condition = color_conditions[(epoch + index) % len(color_conditions)]\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            optim.zero_grad(set_to_none=True)\n            predicted, _ = model(base, xyz, neighbors, rgb)\n            loss = chamfer(predicted, target, knn)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Non-finite training loss for {mode} on {sample.oid}")\n            loss.backward()\n            optim.step()\n            train_losses.append(loss.item())\n        model.eval()\n        val_scores = []\n        with torch.no_grad():\n            for sample in samples["val"]:\n                for condition in color_conditions:\n                    base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n                    predicted, _ = model(base, xyz, neighbors, rgb)\n                    val_scores.append(chamfer(predicted, target, knn).item() * sample.metric_scale**2)\n        score = float(np.mean(val_scores))\n        if not math.isfinite(score):\n            raise RuntimeError(f"Non-finite validation score for {mode}")\n        history.append(dict(epoch=epoch + 1, train_cd=float(np.mean(train_losses)), validation_cd=score))\n        if score < best_score:\n            best_score = score\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n        seconds = time.monotonic() - started\n        print(f"{mode} · epoch {epoch + 1}/{epochs} · validation CD {score:.6g} · {seconds:.1f} s", flush=True)\n    torch.save(best_state, directory / f"{mode}.pt")\n    model.load_state_dict(best_state)\n    return model.eval(), history\n\n\n@torch.no_grad()\ndef evaluate(samples, heads, prepared, device, knn):\n    rows = []\n    for sample in samples:\n        for condition in ("original", "jitter"):\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            predictions = {"noisy": (xyz, None), "frozen_pgd": (base, None)}\n            for mode, head in heads.items():\n                predictions[mode] = head(base, xyz, neighbors, rgb)\n            for method, (predicted, gate) in predictions.items():\n                score = chamfer(predicted, target, knn).item() * sample.metric_scale**2\n                if not math.isfinite(score):\n                    raise RuntimeError(f"Non-finite test score for {method} on {sample.oid}")\n                rows.append(dict(object_id=sample.oid, group=sample.group, xyz_noise=sample.noise,\n                                 rgb=condition, method=method, cd_squared=score,\n                                 gate_mean=gate.mean().item() if method == "gated_rgb" else ""))\n    return rows\n\n\ndef write_csv(path, rows):\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef self_test():\n    torch.manual_seed(42)\n    xyz = torch.randn(32, 3)\n    base = xyz * 0.99\n    rgb = torch.rand_like(xyz)\n    distances = torch.cdist(xyz, xyz)\n    distances.fill_diagonal_(float("inf"))\n    indices = distances.argsort(dim=1)[:, :NEIGHBORS]\n    counts = []\n    for mode in MODES:\n        head = ResidualHead(mode)\n        counts.append(sum(p.numel() for p in head.parameters()))\n        prediction, _ = head(base, xyz, indices, rgb)\n        assert torch.equal(prediction, base)\n        optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3)\n        target = xyz * 0.97 + rgb * 0.05\n        for _ in range(4):\n            optimizer.zero_grad()\n            prediction, _ = head(base, xyz, indices, rgb)\n            (prediction - target).square().mean().backward()\n            assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in head.parameters())\n            optimizer.step()\n        prediction, gate = head(base, xyz, indices, rgb)\n        other, _ = head(base, xyz, indices, 1 - rgb)\n        assert prediction.shape == xyz.shape and torch.isfinite(prediction).all()\n        assert (0 < gate).all() and (gate < 1).all()\n        if mode == "geometry":\n            assert torch.equal(prediction, other)\n        else:\n            assert not torch.equal(prediction, other)\n            constant_a, _ = head(base, xyz, indices, torch.ones_like(rgb))\n            constant_b, _ = head(base, xyz, indices, torch.zeros_like(rgb))\n            assert torch.equal(constant_a, constant_b)\n        if mode == "gated_rgb":\n            assert gate.std() > 0\n        else:\n            assert torch.equal(gate[0].expand_as(gate), gate)\n    assert len(set(counts)) == 1\n    print(f"PASS · all three heads trained · {counts[0]} trainable parameters each")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--self-test", action="store_true")\n    parser.add_argument("--data", type=Path)\n    parser.add_argument("--pgd", type=Path, default=Path("/content/PGD"))\n    parser.add_argument("--out", type=Path)\n    parser.add_argument("--cache", type=Path)\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--iterations", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    if args.self_test:\n        self_test()\n        return\n    if args.data is None or args.out is None or args.epochs < 1 or args.iterations < 1:\n        parser.error("A training run requires --data, --out and positive training settings")\n    if not torch.cuda.is_available():\n        parser.error("The published PGD operators require a CUDA runtime")\n    from pytorch3d.ops import knn_points\n    device = torch.device("cuda")\n    checkpoint = args.pgd / "pretrained" / "PGD.ckpt"\n    samples = load_samples(args.data, args.smoke)\n    args.out.mkdir(parents=True, exist_ok=False)\n    cache = args.cache or args.out / "cache"\n    prepared, config = precompute(samples, args.pgd.resolve(), checkpoint, args.iterations, cache,\n                                  args.seed, device, knn_points)\n    epochs = min(args.epochs, 2) if args.smoke else args.epochs\n    heads, history = {}, {}\n    for mode in MODES:\n        heads[mode], history[mode] = fit(\n            mode, samples, prepared, epochs, args.seed, device, knn_points, args.out,\n        )\n    rows = evaluate(samples["test"], heads, prepared, device, knn_points)\n    write_csv(args.out / "per_object.csv", rows)\n    groups = defaultdict(list)\n    for row in rows:\n        for group in ("all", row["group"]):\n            key = (group, row["xyz_noise"], row["rgb"], row["method"])\n            groups[key].append(row["cd_squared"])\n    write_csv(args.out / "summary.csv", [\n        dict(group=group, xyz_noise=noise, rgb=rgb, method=method, n_objects=len(values),\n             mean_cd_squared=float(np.mean(values)))\n        for (group, noise, rgb, method), values in sorted(groups.items())\n    ])\n    metadata = dict(pgd=config, epochs=epochs, seed=args.seed, smoke=args.smoke,\n                    selection_sha256=digest_file(args.data / "selection.json"), history=history,\n                    objects={split: sorted({s.oid for s in subset}) for split, subset in samples.items()})\n    (args.out / "run.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(f"COMPLETE · {\'smoke\' if args.smoke else \'full\'} · {len(rows)} test rows · {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'task4/evaluate_surface.py': '"""Measure the completed GSO experiment against its original triangle surfaces."""\nimport argparse\nfrom collections import defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport tarfile\n\nimport numpy as np\nimport pandas as pd\nimport point_cloud_utils as pcu\nimport requests\nimport torch\nimport trimesh\nfrom scipy.spatial import cKDTree\n\nfrom train_rgb_pgd import MODES, XYZ_LEVELS, NEIGHBORS, ResidualHead, digest_file, load_cloud\n\n\ndef find_shards(ids, candidates):\n    mapping = defaultdict(set)\n    found = set()\n    for archive in sorted(candidates.glob("*.npz")):\n        with np.load(archive, allow_pickle=False) as data:\n            matching = ids.intersection(data["ids"].tolist())\n        if found.intersection(matching):\n            raise ValueError("Duplicate test IDs in the candidate archives")\n        if matching:\n            mapping[int(archive.stem)].update(matching)\n            found.update(matching)\n        if found == ids:\n            return mapping\n    raise FileNotFoundError(f"Missing source shard for {sorted(ids - found)}")\n\n\ndef original_meshes(source, revision, mapping, directory):\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def retrieve(item):\n        index, objects = item\n        pending = {oid for oid in objects if not (directory / f"{oid}.glb").is_file()}\n        if not pending:\n            return\n        archive = f"gso-train-{index:05d}.tar"\n        url = f"https://huggingface.co/datasets/{source}/resolve/{revision}/data/{archive}"\n        for attempt in range(3):\n            by_stem = defaultdict(dict)\n            try:\n                with requests.get(url, stream=True, timeout=(20, 90)) as response:\n                    response.raise_for_status()\n                    with tarfile.open(fileobj=response.raw, mode="r|*") as tar:\n                        for member in tar:\n                            stem, _, suffix = Path(member.name).name.partition(".")\n                            if not member.isfile() or suffix not in {"glb", "json"}:\n                                continue\n                            if stem not in pending:\n                                continue\n                            content = tar.extractfile(member).read()\n                            entry = by_stem[stem]\n                            entry[suffix] = content\n                            if len(entry) == 2:\n                                oid = str(json.loads(entry["json"])["object_id"])\n                                if oid != stem:\n                                    raise ValueError(f"{archive}: metadata and archive key differ for {stem}")\n                                path = directory / f"{oid}.glb"\n                                partial = path.with_suffix(".part")\n                                partial.write_bytes(entry["glb"])\n                                partial.replace(path)\n                                pending.remove(oid)\n                                del by_stem[stem]\n                                if not pending:\n                                    break\n                if pending:\n                    raise ValueError(f"{archive}: did not find meshes for {sorted(pending)}")\n                print(f"{archive} · {len(objects)} reference mesh(es) ready", flush=True)\n                return\n            except (requests.RequestException, OSError, tarfile.ReadError):\n                if attempt == 2:\n                    raise\n\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        list(pool.map(retrieve, sorted(mapping.items())))\n\n\ndef normalize(xyz):\n    center = (xyz.max(0) + xyz.min(0)) / 2\n    radius = np.linalg.norm(xyz - center, axis=1).max()\n    if radius <= 0 or not np.isfinite(radius):\n        raise ValueError("Degenerate cloud")\n    return center, radius\n\n\ndef point_to_surface(points, vertices, faces):\n    _, fid, bc = pcu.closest_points_on_mesh(np.ascontiguousarray(points), vertices, faces)\n    if np.any(fid < 0) or not np.isfinite(bc).all():\n        raise ValueError("Failed closest-point query")\n    closest = pcu.interpolate_barycentric_coords(faces, fid, bc, vertices)\n    return np.sum((points - closest)**2, axis=-1)\n\n\ndef measure(args):\n    meta = json.loads((args.run / "run.json").read_text())\n    if meta["smoke"] or digest_file(args.data / "selection.json") != meta["selection_sha256"]:\n        raise ValueError("This full run does not match the supplied Task 3 split")\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    selections = json.loads((args.data / "selection.json").read_text())\n    ids = set(meta["objects"]["test"])\n    if ids != {oid for oid, info in selections.items() if info["split"] == "test"} or len(ids) != 12:\n        raise ValueError("Run and dataset disagree about held-out objects")\n    if any(Path(oid).name != oid or oid in {".", ".."} for oid in ids):\n        raise ValueError("Invalid object ID")\n    reference = pd.read_csv(args.run / "per_object.csv").set_index(\n        ["object_id", "xyz_noise", "rgb", "method"], verify_integrity=True\n    )\n    if len(reference) != 360:\n        raise ValueError("Full test CSV must have 360 rows")\n    masks = {}\n    if args.point_masks is not None:\n        with np.load(args.point_masks, allow_pickle=False) as mask_archive:\n            masks = {oid: mask_archive[oid] for oid in mask_archive.files}\n        if not masks or not set(masks) <= ids:\n            raise ValueError("Point masks must name held-out objects")\n        ids = set(masks)\n    mesh_dir = args.mesh_cache / manifest["revision"]\n    original_meshes(manifest["source"], manifest["revision"], find_shards(ids, args.candidates), mesh_dir)\n\n    torch.set_num_threads(2)\n    heads = {}\n    for mode in MODES:\n        head = ResidualHead(mode)\n        head.load_state_dict(torch.load(args.run / f"{mode}.pt", map_location="cpu", weights_only=True))\n        heads[mode] = head.eval()\n    cache_prefix = json.dumps(meta["pgd"], sort_keys=True).encode()\n    rows, mesh_checks = [], []\n    with torch.no_grad():\n        for oid in sorted(ids):\n            folder = args.data / "test" / oid\n            clean, _ = load_cloud(folder / "clean.npz")\n            clean_center, clean_radius = normalize(clean)\n            scene = trimesh.load(io.BytesIO((mesh_dir / f"{oid}.glb").read_bytes()), file_type="glb", force="scene")\n            mesh = trimesh.util.concatenate([m for m in scene.dump() if isinstance(m, trimesh.Trimesh)])\n            vertices = np.ascontiguousarray((mesh.vertices - clean_center) / clean_radius, dtype=np.float64)\n            faces = np.ascontiguousarray(mesh.faces, dtype=np.int32)\n            points_per_cloud = len(clean)\n            mask = masks.get(oid)\n            if mask is not None:\n                if mask.dtype != np.bool_ or mask.shape != (len(clean),) or not mask.any():\n                    raise ValueError(f"Invalid clean-point mask for {oid}")\n                points_per_cloud = int(mask.sum())\n            all_points = [(clean.astype(np.float64) - clean_center) / clean_radius]\n            object_rows = []\n            for noise in XYZ_LEVELS:\n                noisy, original = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                same_noisy, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                if not np.array_equal(noisy, same_noisy):\n                    raise ValueError(f"XYZ differs between RGB settings for {oid}")\n                center, radius = normalize(noisy)\n                xyz = (noisy - center) / radius\n                key = hashlib.sha256(cache_prefix + xyz.tobytes()).hexdigest()\n                pgd = np.load(args.pgd_cache / f"{key}.npy", allow_pickle=False)\n                x = torch.from_numpy(xyz)\n                base = torch.from_numpy(pgd)\n                target = (clean - center) / radius\n                target_tree = cKDTree(target)\n                _, neighbor_ids = cKDTree(xyz).query(xyz, k=NEIGHBORS + 1)\n                neighbors = torch.from_numpy(np.asarray(\n                    [row[row != i][:NEIGHBORS] for i, row in enumerate(neighbor_ids)]\n                ))\n                for condition, rgb in [("original", original), ("jitter", jitter)]:\n                    colors = torch.from_numpy(rgb)\n                    predictions = {"noisy": x, "frozen_pgd": base}\n                    predictions.update({mode: head(base, x, neighbors, colors)[0] for mode, head in heads.items()})\n                    for method, prediction in predictions.items():\n                        saved = reference.loc[(oid, noise, condition, method)]\n                        predicted = prediction.numpy()\n                        a, _ = target_tree.query(predicted)\n                        b, _ = cKDTree(predicted).query(target)\n                        score = (np.mean(a**2) + np.mean(b**2)) * float(radius / clean_radius)**2\n                        if not np.isclose(score, saved.cd_squared, rtol=1e-4, atol=1e-8):\n                            raise ValueError(\n                                f"Recomputed Chamfer mismatch for {oid}/{noise}/{condition}/{method}: "\n                                f"{score:g} vs {saved.cd_squared:g}"\n                            )\n                        row = dict(object_id=oid, group=saved.group, xyz_noise=noise, rgb=condition, method=method)\n                        if mask is None:\n                            row["cd_squared"] = saved.cd_squared\n                        else:\n                            row["n_points"] = points_per_cloud\n                        normalized = (predicted.astype(np.float64) * radius + center - clean_center) / clean_radius\n                        all_points.append(normalized)\n                        object_rows.append(row)\n            if mask is not None:\n                all_points = [points[mask] for points in all_points]\n            squared = point_to_surface(np.concatenate(all_points), vertices, faces)\n            squared = squared.reshape(len(all_points), points_per_cloud)\n            aligned = float(np.sqrt(squared[0].mean()))\n            if aligned > 1e-5 or not np.isfinite(squared).all():\n                raise ValueError(f"Reference mesh misalignment for {oid}: clean RMS = {aligned:g}")\n            mesh_checks.append(dict(object_id=oid, n_faces=len(faces), clean_rms=aligned))\n            for row, distances in zip(object_rows, squared[1:]):\n                row["p2surface_mean"] = float(np.sqrt(distances).mean())\n                row["p2surface_squared"] = float(distances.mean())\n            rows.extend(object_rows)\n            print(f"{len(mesh_checks)}/{len(ids)} · {oid} · clean reference RMS {aligned:.3g}", flush=True)\n    result = pd.DataFrame(rows)\n    expected = reference.loc[reference.index.get_level_values("object_id").isin(ids)]\n    actual = result.set_index(reference.index.names, verify_integrity=True)\n    if not actual.index.sort_values().equals(expected.index.sort_values()):\n        raise ValueError("Surface rows do not cover the selected test cases")\n    metrics = ["p2surface_mean", "p2surface_squared"]\n    if not masks:\n        metrics.insert(0, "cd_squared")\n    args.out.mkdir(parents=True, exist_ok=True)\n    result.to_csv(args.out / "per_object_surface.csv", index=False)\n    pd.DataFrame(mesh_checks).to_csv(args.out / "reference_mesh_checks.csv", index=False)\n    result.groupby(["xyz_noise", "rgb", "method"])[metrics].mean().to_csv(args.out / "surface_by_condition.csv")\n    result.groupby(["group", "method"])[metrics].mean().to_csv(args.out / "surface_by_group.csv")\n    print("Overall means:\\n" + result.groupby("method")[metrics].mean().to_string(float_format="%.9g"))\n    print("Group means, squared P2S × 1e6:\\n" +\n          (result.groupby(["group", "method"]).p2surface_squared.mean().unstack()*1e6).to_string(float_format="%.6g"))\n    print(f"COMPLETE · {len(result)} surface scores · {args.out}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "run", "pgd-cache", "candidates", "mesh-cache", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    parser.add_argument("--point-masks", type=Path, help="NPZ mapping held-out object IDs to clean-point masks")\n    measure(parser.parse_args())\n', 'task4/texture_masks.py': '"""Recreate the original GSO texture-only masks from the Task 3 candidates."""\nimport argparse\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nfrom threadpoolctl import threadpool_limits\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "data"))\nfrom ranking import compute_scores, compute_thresholds\n\nCANDIDATE_COUNT = 1030\nSHARD_COUNT = 43\nNEIGHBOURS = 24\n\n\ndef generate(data, candidates, out):\n    manifest = json.loads((data / "manifest.json").read_text())\n    selection = json.loads((data / "selection.json").read_text())\n    targets = {\n        oid for oid, item in selection.items()\n        if item["split"] == "test" and item["group"] == "texture_only"\n    }\n    if len(targets) != 2 or manifest["skipped"]:\n        raise ValueError("Expected the original complete GSO run and its two texture-only test shapes")\n    shards = sorted(candidates.glob("*.npz"))\n    if len(shards) != SHARD_COUNT:\n        raise ValueError(f"Expected {SHARD_COUNT} Task 3 candidate shards in {candidates}")\n\n    scores = {}\n    with threadpool_limits(limits=2):\n        for index, shard in enumerate(shards, 1):\n            with np.load(shard, allow_pickle=False) as archive:\n                if len(archive["skipped"]):\n                    raise ValueError(f"Missing candidate objects in {shard}")\n                for name, xyz, rgb in zip(archive["ids"], archive["xyz"], archive["rgb"], strict=True):\n                    oid = str(name)\n                    if oid in scores:\n                        raise ValueError(f"Duplicate candidate object ID {oid}")\n                    g, c = compute_scores(xyz, rgb, k=NEIGHBOURS)\n                    if not (np.isfinite(g).all() and np.isfinite(c).all()):\n                        raise ValueError(f"Invalid candidate scores for {oid}")\n                    scores[oid] = (g, c)\n                    if oid in targets:\n                        with np.load(data / "test" / oid / "clean.npz", allow_pickle=False) as clean:\n                            if not (np.array_equal(clean["xyz"], xyz) and np.array_equal(clean["rgb"], rgb)):\n                                raise ValueError(f"Candidate and clean test cloud differ for {oid}")\n            if index % 7 == 0 or index == len(shards):\n                print(f"{index}/{len(shards)} candidate archives · {len(scores)} objects", flush=True)\n\n    if len(scores) != CANDIDATE_COUNT or not targets <= scores.keys():\n        raise ValueError("Incomplete Task 3 candidate cache")\n    thresholds = compute_thresholds(scores)\n    geometry_p50 = float(thresholds["g_low"])\n    colour_p90 = float(thresholds["c_high"])\n    masks = {oid: (scores[oid][0] <= geometry_p50) & (scores[oid][1] >= colour_p90)\n             for oid in sorted(targets)}\n    if not all(mask.any() for mask in masks.values()):\n        raise ValueError("Texture-only mask is empty")\n\n    metadata = dict(revision=manifest["revision"], candidate_objects=len(scores), neighbours=NEIGHBOURS,\n                    geometry_p50=geometry_p50, colour_p90=colour_p90,\n                    selected_points={oid: int(mask.sum()) for oid, mask in masks.items()},\n                    numpy=version("numpy"), scikit_learn=version("scikit-learn"))\n    out.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(out / "clean_point_masks.npz", **masks)\n    (out / "mask_metadata.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(json.dumps(metadata, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "candidates", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    args = parser.parse_args()\n    generate(args.data, args.candidates, args.out)\n', 'task5/evaluate_gso.py': '"""Task 5 paired RGB interventions on the held-out GSO benchmark."""\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport trimesh\nfrom scipy.spatial import cKDTree\nfrom sklearn.neighbors import NearestNeighbors\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "task4"))\nfrom evaluate_surface import find_shards, normalize, original_meshes, point_to_surface\nfrom train_rgb_pgd import MODES, NEIGHBORS, XYZ_LEVELS, ResidualHead, digest_file, load_cloud\n\nCOLOURS = ("original", "constant", "shuffled", "gaussian_0.05", "gaussian_0.15")\nMETHODS = ("noisy", "frozen_pgd", *MODES)\nBAND_RADIUS = 0.03\nMIN_REGION_POINTS = 30\nMIN_TEXTURE_LINKS = 30\nSEED = 20261123\n\n\ndef seed_for(oid, purpose):\n    return int.from_bytes(hashlib.sha256(f"{SEED}:{oid}:{purpose}".encode()).digest()[:8], "little")\n\n\ndef intervention_colors(rgb, jitter, oid):\n    perm = np.random.default_rng(seed_for(oid, "shuffle")).permutation(len(rgb))\n    noise = np.random.default_rng(seed_for(oid, "gaussian")).normal(0, 0.15, rgb.shape)\n    return {\n        "original": rgb,\n        "constant": np.broadcast_to(rgb.mean(axis=0), rgb.shape).copy(),\n        "shuffled": rgb[perm],\n        "gaussian_0.05": jitter,\n        "gaussian_0.15": np.clip(rgb + noise, 0, 1).astype(np.float32),\n    }\n\n\ndef geometry_regions(clean, rgb, mesh, g_median, c_p90):\n    xyz = np.ascontiguousarray(clean, dtype=np.float64)\n    edges = mesh.face_adjacency_edges\n    angles = mesh.face_adjacency_angles\n    segments = mesh.vertices[edges[(angles >= np.deg2rad(30)) & (angles <= np.deg2rad(150))]]\n    if len(segments):\n        n_samples = np.ceil(np.linalg.norm(segments[:, 1] - segments[:, 0], axis=1) / 0.005).astype(int)\n        divisions = np.maximum(n_samples, 1)\n        segment_id = np.repeat(np.arange(len(segments)), divisions + 1)\n        first_index = np.repeat(np.cumsum(divisions + 1) - divisions - 1, divisions + 1)\n        steps = (np.arange(len(segment_id)) - first_index) / divisions[segment_id]\n        edge_samples = segments[segment_id, 0] + steps[:, None] * (segments[segment_id, 1] - segments[segment_id, 0])\n        sharp = cKDTree(edge_samples).query(xyz)[0] <= BAND_RADIUS\n    else:\n        sharp = np.zeros(len(xyz), dtype=bool)\n\n    knn = NearestNeighbors(n_neighbors=25, algorithm="kd_tree").fit(xyz)\n    indices = knn.kneighbors(xyz, return_distance=False)[:, 1:]\n    neighbours = xyz[indices]\n    centered = neighbours - neighbours.mean(axis=1, keepdims=True)\n    eigen = np.linalg.eigvalsh(np.einsum("nki,nkj->nij", centered, centered) / indices.shape[1])\n    g = eigen[:, 0] / (eigen.sum(axis=1) + 1e-12)\n    c = np.linalg.norm(rgb[indices] - rgb[:, None, :], axis=2).mean(axis=1)\n    texture = (g <= g_median) & (c >= c_p90)\n\n    import point_cloud_utils as pcu\n\n    vertices = np.ascontiguousarray(mesh.vertices, dtype=np.float64)\n    faces = np.ascontiguousarray(mesh.faces, dtype=np.int32)\n    _, face_ids, _ = pcu.closest_points_on_mesh(xyz, vertices, faces)\n    normals = mesh.face_normals[face_ids]\n    left = np.repeat(np.arange(len(xyz)), indices.shape[1])\n    right = indices.ravel()\n    potential = (texture[left] | texture[right]) & ~sharp[left] & ~sharp[right]\n    pairs = np.unique(np.sort(np.column_stack((left[potential], right[potential])), axis=1), axis=0)\n    a, b = pairs.T\n    vector = xyz[b] - xyz[a]\n    pair_normal = normals[a] + normals[b]\n    pair_normal /= np.maximum(np.linalg.norm(pair_normal, axis=1, keepdims=True), 1e-12)\n    clean_step = np.einsum("ij,ij->i", vector, pair_normal)\n    across_texture = (\n        (np.linalg.norm(rgb[b] - rgb[a], axis=1) >= c_p90)\n        & (np.einsum("ij,ij->i", normals[a], normals[b]) >= np.cos(np.deg2rad(15)))\n        & (np.linalg.norm(vector, axis=1) <= 0.06)\n        & (np.abs(clean_step) <= 0.003)\n    )\n    a, b = a[across_texture], b[across_texture]\n    return sharp, texture, a, b, pair_normal[across_texture], clean_step[across_texture], normals\n\n\ndef read_mesh(path, center, radius):\n    scene = trimesh.load(io.BytesIO(path.read_bytes()), file_type="glb", force="scene")\n    original = trimesh.util.concatenate([part for part in scene.dump() if isinstance(part, trimesh.Trimesh)])\n    raw = trimesh.Trimesh(vertices=(original.vertices - center) / radius, faces=original.faces, process=False)\n    topology = raw.copy()\n    topology.merge_vertices()\n    return raw, topology\n\n\ndef measured_output(pred, surface_sq, sharp, texture, left, right, pair_normal, clean_step, clean, clean_tree):\n    pred_tree = cKDTree(pred)\n    forward = clean_tree.query(pred)[0]\n    backward = pred_tree.query(clean)[0]\n    stats = {\n        "cd_squared": float(np.mean(forward**2) + np.mean(backward**2)),\n        "p2surface_squared": float(surface_sq.mean()),\n        "p2surface_mean": float(np.sqrt(surface_sq).mean()),\n    }\n    if sharp.sum() >= MIN_REGION_POINTS:\n        stats["sharp_p2surface_squared"] = float(surface_sq[sharp].mean())\n        stats["sharp_coverage_squared"] = float(np.mean(backward[sharp]**2))\n        stats["sharp_coverage_p95"] = float(np.quantile(backward[sharp], 0.95))\n    if texture.sum() >= MIN_REGION_POINTS:\n        stats["texture_p2surface_squared"] = float(surface_sq[texture].mean())\n    if len(left) >= MIN_TEXTURE_LINKS:\n        output_step = np.einsum("ij,ij->i", pred[right] - pred[left], pair_normal)\n        stats["texture_normal_jump_rms"] = float(np.sqrt(np.mean((output_step - clean_step)**2)))\n    return stats\n\n\ndef measure(args):\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    meta = json.loads((args.run / "run.json").read_text())\n    selection = json.loads((args.data / "selection.json").read_text())\n    assert digest_file(args.data / "selection.json") == meta["selection_sha256"]\n    ids = sorted(meta["objects"]["test"])\n    assert len(ids) == 12 and set(ids) == {oid for oid, entry in selection.items() if entry["split"] == "test"}\n    thresholds = json.loads((args.run / "surface/texture_regions/mask_metadata.json").read_text())\n    assert thresholds["revision"] == manifest["revision"] and thresholds["candidate_objects"] == 1030\n    reference = pd.read_csv(args.run / "surface/per_object_surface.csv").set_index(\n        ["object_id", "xyz_noise", "rgb", "method"], verify_integrity=True\n    )\n    mesh_dir = args.mesh_cache / manifest["revision"]\n    original_meshes(manifest["source"], manifest["revision"], find_shards(set(ids), args.candidates), mesh_dir)\n    prefix = json.dumps(meta["pgd"], sort_keys=True).encode()\n    torch.set_num_threads(2)\n    heads = {}\n    for mode in MODES:\n        head = ResidualHead(mode)\n        head.load_state_dict(torch.load(args.run / f"{mode}.pt", map_location="cpu", weights_only=True))\n        heads[mode] = head.eval()\n    args.out.mkdir(parents=True, exist_ok=True)\n    picture_dir = args.out / "cases"\n    picture_dir.mkdir(exist_ok=True)\n    results, region_rows = [], []\n    with torch.inference_mode():\n        for oid in ids:\n            folder = args.data / "test" / oid\n            clean, clean_rgb = load_cloud(folder / "clean.npz")\n            center, radius = normalize(clean)\n            norm_clean = (clean.astype(np.float64) - center) / radius\n            tree = cKDTree(norm_clean)\n            raw_mesh, topology = read_mesh(mesh_dir / f"{oid}.glb", center, radius)\n            sharp, texture, left, right, pair_normal, clean_step, normals = geometry_regions(\n                norm_clean, clean_rgb, topology, thresholds["geometry_p50"], thresholds["colour_p90"]\n            )\n            vertices = np.ascontiguousarray(raw_mesh.vertices, dtype=np.float64)\n            faces = np.ascontiguousarray(raw_mesh.faces, dtype=np.int32)\n            clean_rms = float(np.sqrt(point_to_surface(norm_clean, vertices, faces).mean()))\n            assert clean_rms < 1e-5, (oid, clean_rms)\n            group = selection[oid]["group"]\n            region_rows.append(dict(object_id=oid, group=group, clean_rms=clean_rms,\n                                    sharp_points=int(sharp.sum()), texture_points=int(texture.sum()), texture_links=len(left)))\n            for noise in XYZ_LEVELS:\n                noisy, rgb = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                same_noisy, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                assert np.array_equal(noisy, same_noisy) and np.array_equal(rgb, clean_rgb)\n                work_center, work_radius = normalize(noisy)\n                xyz = (noisy - work_center) / work_radius\n                key = hashlib.sha256(prefix + xyz.tobytes()).hexdigest()\n                baseline = np.load(args.pgd_cache / f"{key}.npy", allow_pickle=False)\n                assert baseline.shape == xyz.shape and np.isfinite(baseline).all()\n                ids_knn = cKDTree(xyz).query(xyz, k=NEIGHBORS + 1)[1]\n                knn = torch.from_numpy(np.asarray([row[row != i][:NEIGHBORS] for i, row in enumerate(ids_knn)]))\n                features = (torch.from_numpy(baseline), torch.from_numpy(xyz), knn)\n                colors = intervention_colors(rgb, jitter, oid)\n                invariant = {"noisy": xyz, "frozen_pgd": baseline, "geometry": heads["geometry"](*features, torch.from_numpy(rgb))[0].numpy()}\n                preds = {("", method): pred for method, pred in invariant.items()}\n                for condition, color in colors.items():\n                    for mode in ("plain_rgb", "gated_rgb"):\n                        preds[(condition, mode)] = heads[mode](*features, torch.from_numpy(color))[0].numpy()\n                normalized = {(rgb_name, method): (pred.astype(np.float64) * work_radius + work_center - center) / radius\n                              for (rgb_name, method), pred in preds.items()}\n                distances = point_to_surface(np.concatenate(list(normalized.values())), vertices, faces).reshape(len(preds), -1)\n                scores = {key: measured_output(pred, surface_sq, sharp, texture, left, right, pair_normal,\n                                               clean_step, norm_clean, tree)\n                          for (key, pred), surface_sq in zip(normalized.items(), distances, strict=True)}\n                for condition in COLOURS:\n                    for method in METHODS:\n                        source = ("", method) if method in invariant else (condition, method)\n                        row = scores[source]\n                        if condition in ("original", "gaussian_0.05"):\n                            color_key = "original" if condition == "original" else "jitter"\n                            previous = reference.loc[(oid, noise, color_key, method)]\n                            for metric in ("cd_squared", "p2surface_squared"):\n                                assert np.isclose(row[metric], previous[metric], rtol=1e-4, atol=1e-8), (oid, method, metric)\n                        results.append(dict(object_id=oid, group=group, xyz_noise=noise, colour=condition,\n                                            method=method, **row))\n                if group == "texture_only" and noise == 0.02:\n                    np.savez_compressed(picture_dir / f"{oid}.npz", clean=norm_clean, rgb=clean_rgb, normals=normals,\n                                        sharp=sharp, texture=texture, left=left, right=right, pair_normal=pair_normal,\n                                        clean_step=clean_step,\n                                        **{f"{color}_{model}": pred for (color, model), pred in normalized.items()})\n            print(f"Task 5 {len(region_rows)}/{len(ids)} · {oid} · {int(sharp.sum())} sharp samples · {len(left)} texture links", flush=True)\n    scores = pd.DataFrame(results)\n    assert len(scores) == 12 * 3 * len(COLOURS) * len(METHODS)\n    assert not scores.duplicated(["object_id", "xyz_noise", "colour", "method"]).any()\n    scores.to_csv(args.out / "scores.csv", index=False)\n    pd.DataFrame(region_rows).to_csv(args.out / "region_sizes.csv", index=False)\n    (args.out / "protocol.json").write_text(json.dumps(dict(seed=SEED, colour_conditions=COLOURS, methods=METHODS,\n        sharp_dihedral_degrees=[30, 150], sharp_band=BAND_RADIUS, edge_sample_spacing=0.005,\n        min_region_points=MIN_REGION_POINTS, min_texture_links=MIN_TEXTURE_LINKS,\n        max_texture_neighbour_distance=0.06, max_clean_normal_jump=0.003,\n        max_texture_normal_angle_degrees=15, mask_thresholds=thresholds, test_ids=ids,\n        run_selection_sha256=meta["selection_sha256"]), indent=2))\n    print(f"TASK 5 SCORES SAVED · {len(scores)} rows · {args.out}", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "run", "pgd-cache", "candidates", "mesh-cache", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    measure(parser.parse_args())\n', 'task5/plane_experts.py': '"""Compare RGB-guided neighbourhood selection with a geometry-only plane expert."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.spatial import cKDTree\nfrom scipy.special import expit\n\nfrom evaluate_gso import (COLOURS, geometry_regions, intervention_colors, load_cloud, measured_output,\n                          normalize, original_meshes, find_shards, point_to_surface, read_mesh, seed_for,\n                          XYZ_LEVELS, digest_file)\n\nK = 32\nRGB_BANDWIDTH = 0.20\nTRAIN_POINTS = 320\nMETHODS = ("frozen_pgd", "geometry_plane", "plain_rgb_plane", "gated_rgb_plane")\n\n\ndef neighbours(points, k=K):\n    distances, indices = cKDTree(points).query(points, k=k + 1)\n    keep = indices != np.arange(len(points))[:, None]\n    column = np.argsort(~keep, axis=1, kind="stable")[:, :k]\n    return np.take_along_axis(distances, column, axis=1), np.take_along_axis(indices, column, axis=1)\n\n\ndef local_plane(points, nearby, weights):\n    weights = weights / weights.sum(axis=1, keepdims=True)\n    center = np.einsum("nk,nkj->nj", weights, nearby)\n    relative = nearby - center[:, None]\n    covariance = np.einsum("nk,nki,nkj->nij", weights, relative, relative, optimize=True)\n    eigen, basis = np.linalg.eigh(covariance)\n    normal = basis[:, :, 0]\n    variation = np.maximum(eigen[:, 0], 0) / np.maximum(eigen.sum(axis=1), 1e-12)\n    offset = np.einsum("ni,ni->n", center - points, normal)\n    return np.clip(offset, -0.04, 0.04)[:, None] * normal, normal, variation, 1 / np.sum(weights**2, axis=1)\n\n\ndef geometric_features(pgd, noisy):\n    distance, ids = neighbours(pgd)\n    nearby = pgd[ids]\n    bandwidth = np.maximum(np.median(distance, axis=1), 1e-6)\n    weights = np.exp(-0.5 * (distance / bandwidth[:, None])**2)\n    delta, normal, variation, support = local_plane(pgd, nearby, weights)\n    features = np.column_stack((np.log1p(100 * variation), np.log1p(100 * np.linalg.norm(delta, axis=1)),\n                                np.log1p(100 * bandwidth), np.log1p(100 * np.linalg.norm(pgd - noisy, axis=1))))\n    return features, delta, normal, variation, (nearby, ids, weights, support)\n\n\ndef colour_features(pgd, rgb, geom, point_rgb):\n    features, delta, normal, variation, (nearby, ids, weights, geometry_support) = geom\n    gap = np.linalg.norm(rgb[ids] - point_rgb[:, None, :], axis=-1)\n    affinity = np.exp(-0.5 * (gap / RGB_BANDWIDTH)**2)\n    weighted = np.maximum(affinity, 1e-8) * weights\n    color_delta, color_normal, color_variation, color_support = local_plane(pgd, nearby, weighted)\n    reliability = np.minimum(color_support / 8, 1)\n    color_delta = delta + reliability[:, None] * (color_delta - delta)\n    extra = np.column_stack((gap.mean(axis=1), color_support / geometry_support,\n                             np.log1p(100 * color_variation) - np.log1p(100 * variation),\n                             1 - np.abs(np.einsum("ni,ni->n", normal, color_normal)),\n                             np.log1p(100 * np.linalg.norm(color_delta - delta, axis=1))))\n    return np.concatenate([features, extra], axis=1), color_delta\n\n\ndef projected(v, n):\n    return np.einsum("ij,ij->i", v, n)\n\n\ndef learn(features, error, delta):\n    center = features.mean(axis=0)\n    scale = np.maximum(features.std(axis=0), 1e-6)\n    standardized = np.clip((features - center) / scale, -8, 8)\n    matrix = np.column_stack((np.ones(len(features)), standardized))\n    count = len(matrix)\n\n    def objective(params):\n        gain = expit(matrix @ params)\n        residual = error + gain * delta\n        risk = np.mean(residual**2) * 1e6 + 0.03 * np.sum(params[1:]**2)\n        gradient = (2e6 / count) * matrix.T @ (residual * delta * gain * (1 - gain))\n        gradient[1:] += 0.06 * params[1:]\n        return risk, gradient\n\n    initial = np.zeros(matrix.shape[1])\n    initial[0] = -1\n    result = minimize(objective, initial, jac=True, method="L-BFGS-B",\n                      bounds=[(-9, 9)] * len(initial), options={"maxiter": 100})\n    if not result.success or not np.isfinite(result.x).all():\n        raise RuntimeError(result.message)\n    return dict(center=center.tolist(), scale=scale.tolist(), params=result.x.tolist())\n\n\ndef gain(model, features):\n    standardized = np.clip((features - model["center"]) / model["scale"], -8, 8)\n    params = np.asarray(model["params"])\n    return expit(params[0] + standardized @ params[1:])\n\n\ndef load_samples(data, run, meta, split, shape):\n    prefix = json.dumps(meta["pgd"], sort_keys=True).encode()\n    path = data / split / shape\n    clean, clean_rgb = load_cloud(path / "clean.npz")\n    clean_center, clean_radius = normalize(clean)\n    norm_clean = (clean.astype(np.float64) - clean_center) / clean_radius\n    for noise in XYZ_LEVELS:\n        noisy, rgb = load_cloud(path / f"noise_xyz_{noise:g}_rgb_0.npz")\n        other, jitter = load_cloud(path / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n        assert np.array_equal(noisy, other) and np.array_equal(rgb, clean_rgb)\n        center, radius = normalize(noisy)\n        xyz = (noisy - center) / radius\n        key = hashlib.sha256(prefix + xyz.tobytes()).hexdigest()\n        baseline = np.load(run.parent / "pgd_cache" / f"{key}.npy", allow_pickle=False)\n        pgd = (baseline.astype(np.float64) * radius + center - clean_center) / clean_radius\n        work_noisy = (noisy.astype(np.float64) - clean_center) / clean_radius\n        yield noise, norm_clean, pgd, work_noisy, intervention_colors(rgb, jitter, shape)\n\n\ndef training_data(data, run, meta):\n    samples = {key: [] for key in ("geometry", "colour", "error", "delta_geo", "delta_rgb")}\n    for i, shape in enumerate(sorted(meta["objects"]["train"]), 1):\n        normals = None\n        for noise, clean, pgd, noisy, colors in load_samples(data, run, meta, "train", shape):\n            if normals is None:\n                _, clean_ids = neighbours(clean, k=24)\n                _, normals, _, _ = local_plane(clean, clean[clean_ids], np.ones(clean_ids.shape))\n            context = geometric_features(pgd, noisy)\n            rng = np.random.default_rng(seed_for(shape, f"fit-{noise:g}"))\n            keep = rng.choice(len(pgd), min(TRAIN_POINTS, len(pgd)), replace=False)\n            sampled = tuple(array[keep] for array in context[:4]) + (tuple(array[keep] for array in context[4]),)\n            features, geometry_delta = sampled[:2]\n            normal = normals[keep]\n            error = projected((pgd - clean)[keep], normal)\n            geo = projected(geometry_delta, normal)\n            for rgb in colors.values():\n                colour, rgb_delta = colour_features(pgd[keep], rgb, sampled, rgb[keep])\n                for key, value in (("geometry", features), ("colour", colour), ("error", error),\n                                   ("delta_geo", geo), ("delta_rgb", projected(rgb_delta, normal))):\n                    samples[key].append(value)\n        if i % 7 == 0:\n            print(f"Plane experts: prepared {i}/42 training objects", flush=True)\n    return {key: np.concatenate(value) for key, value in samples.items()}\n\n\ndef fit(args, meta):\n    x = training_data(args.data, args.run, meta)\n    geom = learn(x["geometry"], x["error"], x["delta_geo"])\n    strength = gain(geom, x["geometry"])\n    residual = x["error"] + strength * x["delta_geo"]\n    opt = strength * (x["delta_rgb"] - x["delta_geo"])\n    global_mix = float(np.clip(-np.dot(residual, opt) / max(np.dot(opt, opt), 1e-12), 0, 1))\n    gate = learn(x["colour"], residual, opt)\n    return dict(geometry=geom, plain_rgb_weight=global_mix, rgb_gate=gate, neighbours=K,\n                rgb_bandwidth=RGB_BANDWIDTH, train_points_per_xyz=TRAIN_POINTS,\n                loss="squared clean-sample normal residual", train_ids=meta["objects"]["train"])\n\n\ndef measure(args, meta, model):\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    selection = json.loads((args.data / "selection.json").read_text())\n    thresholds = json.loads((args.run / "surface/texture_regions/mask_metadata.json").read_text())\n    ids = set(meta["objects"]["test"])\n    original_meshes(manifest["source"], manifest["revision"], find_shards(ids, args.candidates),\n                    args.mesh_cache / manifest["revision"])\n    rows, val_rows = [], []\n    for split, ids in (("val", meta["objects"]["val"]), ("test", meta["objects"]["test"])):\n        for shape in sorted(ids):\n            if split == "test":\n                ref, rgb = load_cloud(args.data / "test" / shape / "clean.npz")\n                center_ref, radius_ref = normalize(ref)\n                mesh, topology = read_mesh(args.mesh_cache / manifest["revision"] / f"{shape}.glb", center_ref, radius_ref)\n                vertices = np.ascontiguousarray(mesh.vertices, dtype=np.float64)\n                faces = np.ascontiguousarray(mesh.faces, dtype=np.int32)\n                sharp, texture, left, right, normal, clean_step, _ = geometry_regions(\n                    (ref.astype(np.float64) - center_ref) / radius_ref, rgb, topology,\n                    thresholds["geometry_p50"], thresholds["colour_p90"])\n            for noise, clean, pgd, noisy, colors in load_samples(args.data, args.run, meta, split, shape):\n                clean_tree = cKDTree(clean)\n                context = geometric_features(pgd, noisy)\n                geo_features, geo_delta = context[:2]\n                strength = gain(model["geometry"], geo_features)[:, None]\n                geometry = pgd + strength * geo_delta\n                for name, rgb in colors.items():\n                    color_features, color_delta = colour_features(pgd, rgb, context, rgb)\n                    difference = strength * (color_delta - geo_delta)\n                    color_gate = gain(model["rgb_gate"], color_features)\n                    pred = [pgd, geometry, geometry + model["plain_rgb_weight"] * difference,\n                            geometry + color_gate[:, None] * difference]\n                    if split == "test":\n                        sq = point_to_surface(np.concatenate(pred), vertices, faces).reshape(len(pred), -1)\n                    for j, (method, value) in enumerate(zip(METHODS, pred, strict=True)):\n                        row = dict(object_id=shape, group=selection[shape]["group"], xyz_noise=noise,\n                                   colour=name, method=method)\n                        if split == "test":\n                            row.update(measured_output(value, sq[j], sharp, texture, left, right, normal,\n                                                       clean_step, clean, clean_tree))\n                            if method == "gated_rgb_plane":\n                                row["mean_rgb_gate"] = float(color_gate.mean())\n                            rows.append(row)\n                        else:\n                            forward = clean_tree.query(value)[0]\n                            backward = cKDTree(value).query(clean)[0]\n                            row["cd_squared"] = float(np.mean(forward**2) + np.mean(backward**2))\n                            val_rows.append(row)\n            print(f"Plane expert {split}: {shape}", flush=True)\n    scores = pd.DataFrame(rows)\n    assert len(scores) == 12 * len(XYZ_LEVELS) * len(COLOURS) * len(METHODS)\n    scores.to_csv(args.out / "test_scores.csv", index=False)\n    pd.DataFrame(val_rows).to_csv(args.out / "validation_scores.csv", index=False)\n    print(f"PLANE EXPERTS COMPLETE · {len(scores)} test rows · {args.out}", flush=True)\n\n\ndef main(args):\n    args.out.mkdir(parents=True, exist_ok=True)\n    meta = json.loads((args.run / "run.json").read_text())\n    assert digest_file(args.data / "selection.json") == meta["selection_sha256"]\n    assert {key: len(meta["objects"][key]) for key in ("train", "val", "test")} == {"train": 42, "val": 6, "test": 12}\n    config_path = args.out / "fit.json"\n    model = fit(args, meta)\n    config_path.write_text(json.dumps(model, indent=2))\n    print(f"Fitted plane experts: plain RGB weight {model[\'plain_rgb_weight\']:.4f}", flush=True)\n    measure(args, meta, model)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "run", "candidates", "mesh-cache", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    main(parser.parse_args())\n', 'task5/paint_probe.py': '"""Add a colour boundary to a known flat GSO patch, with paired geometry."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.spatial import cKDTree\n\nfrom evaluate_gso import (NEIGHBORS, XYZ_LEVELS, ResidualHead, geometry_regions, load_cloud,\n                          normalize, read_mesh, seed_for)\nfrom plane_experts import geometric_features, colour_features, gain\n\nRADIUS = 0.24\nNORMAL_ANGLE = np.cos(np.deg2rad(8))\nCONTRASTS = (0.0, 0.40, 0.80)\n\n\ndef select_patch(xyz, normals, shape):\n    tree = cKDTree(xyz)\n    rng = np.random.default_rng(seed_for(shape, "paint_probe"))\n    pool = rng.choice(len(xyz), min(400, len(xyz)), replace=False)\n    # Select by true geometry and spatial coverage only, before model outputs.\n    best = None\n    for index, group in zip(pool, tree.query_ball_point(xyz[pool], RADIUS), strict=True):\n        n = normals[index]\n        local = xyz[group] - xyz[index]\n        clean_plane = (np.abs(local @ n) <= 0.002) & ((normals[group] @ n) >= NORMAL_ANGLE)\n        if clean_plane.sum() < 55:\n            continue\n        group = np.asarray(group)[clean_plane]\n        local = xyz[group] - xyz[index]\n        axis = np.eye(3)[np.argmin(np.abs(n))]\n        u = np.cross(n, axis)\n        u /= np.linalg.norm(u)\n        v = np.cross(n, u)\n        coordinates = np.column_stack((local @ u, local @ v))\n        s, t = coordinates.T\n        left = (s < 0) & (s >= -0.055) & (np.abs(t) <= 0.13)\n        right = (s >= 0) & (s <= 0.055) & (np.abs(t) <= 0.13)\n        if min(left.sum(), right.sum()) < 11:\n            continue\n        gap, nearest = cKDTree(coordinates[right]).query(coordinates[left])\n        nearby = gap <= 0.067\n        pairs = np.column_stack((group[left][nearby], group[right][nearest[nearby]]))\n        if len(pairs) < 11 or len(np.unique(pairs[:, 1])) < 8:\n            continue\n        priority = (min(len(np.unique(pairs[:, 0])), len(np.unique(pairs[:, 1]))), len(pairs))\n        if best is None or priority > best[0]:\n            best = (priority, group, coordinates, pairs, n, int(index))\n    return None if best is None else best[1:]\n\n\ndef run(args):\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    meta = json.loads((args.run / "run.json").read_text())\n    threshold = json.loads((args.run / "surface/texture_regions/mask_metadata.json").read_text())\n    prefix = json.dumps(meta["pgd"], sort_keys=True).encode()\n    plane_model = json.loads(args.plane_model.read_text()) if args.plane_model.is_file() else None\n    torch.set_num_threads(2)\n    heads = {name: ResidualHead(name) for name in ("plain_rgb", "gated_rgb")}\n    for name, network in heads.items():\n        network.load_state_dict(torch.load(args.run / f"{name}.pt", map_location="cpu", weights_only=True))\n        network.eval()\n    rows = []\n    cases = args.out / "paint_cases"\n    cases.mkdir(parents=True, exist_ok=True)\n    with torch.inference_mode():\n        for shape in sorted(meta["objects"]["test"]):\n            folder = args.data / "test" / shape\n            xyz, rgb = load_cloud(folder / "clean.npz")\n            center, radius = normalize(xyz)\n            clean = (xyz.astype(np.float64) - center) / radius\n            _, topology = read_mesh(args.mesh_cache / manifest["revision"] / f"{shape}.glb", center, radius)\n            normals = geometry_regions(clean, rgb, topology, threshold["geometry_p50"], threshold["colour_p90"])[-1]\n            selected = select_patch(clean, normals, shape)\n            if selected is None:\n                print(f"Paint probe: no qualifying flat patch for {shape}", flush=True)\n                continue\n            area, local, pairs, normal, anchor = selected\n            left, right = pairs.T\n            clean_step = (clean[right] - clean[left]) @ normal\n            cross = np.unique(pairs)\n            for noise in XYZ_LEVELS:\n                noisy, _ = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                noise_center, noise_radius = normalize(noisy)\n                norm_noisy = (noisy - noise_center) / noise_radius\n                pgd_key = hashlib.sha256(prefix + norm_noisy.tobytes()).hexdigest()\n                pgd = np.load(args.run.parent / "pgd_cache" / f"{pgd_key}.npy", allow_pickle=False)\n                _, ids = cKDTree(norm_noisy).query(norm_noisy, k=NEIGHBORS + 1)\n                ids = torch.from_numpy(np.asarray([row[row != i][:NEIGHBORS] for i, row in enumerate(ids)]))\n                features = (torch.from_numpy(pgd), torch.from_numpy(norm_noisy), ids)\n\n                def to_clean(points):\n                    return (points.astype(np.float64) * noise_radius + noise_center - center) / radius\n\n                base = to_clean(pgd)\n                if plane_model is not None:\n                    geom = geometric_features(base, to_clean(norm_noisy))\n                    geo_delta = geom[1]\n                    strength = gain(plane_model["geometry"], geom[0])[:, None]\n                    geo = base + strength * geo_delta\n                constants, heights = {}, {}\n                for contrast in CONTRASTS:\n                    colors = np.full(rgb.shape, 0.5, dtype=np.float32)\n                    colors[area] += (0.5 * contrast * np.where(local[:, 0] >= 0, 1, -1))[:, None]\n                    pred = {"frozen_pgd": base}\n                    for name, network in heads.items():\n                        pred[name] = to_clean(network(*features, torch.from_numpy(colors))[0].numpy())\n                    if plane_model is not None:\n                        cues, color_delta = colour_features(base, colors, geom, colors)\n                        delta = strength * (color_delta - geo_delta)\n                        pred["geometry_plane"] = geo\n                        pred["plain_rgb_plane"] = geo + plane_model["plain_rgb_weight"] * delta\n                        pred["gated_rgb_plane"] = geo + gain(plane_model["rgb_gate"], cues)[:, None] * delta\n                    for method, values in pred.items():\n                        step = (values[right] - values[left]) @ normal - clean_step\n                        height = (values[cross] - clean[cross]) @ normal\n                        if contrast == 0:\n                            constants[method] = (values.copy(), step.copy())\n                        ref, ref_step = constants[method]\n                        effects = (values[cross] - ref[cross]) @ normal\n                        rows.append(dict(object_id=shape, clean_anchor=anchor, xyz_noise=noise, paint_contrast=contrast,\n                                         method=method, n_patch_points=len(area), n_links=len(pairs),\n                                         normal_height_rms=float(np.sqrt(np.mean(height**2))),\n                                         step_rms=float(np.sqrt(np.mean(step**2))),\n                                         step_rms_change=float(np.sqrt(np.mean(step**2)) - np.sqrt(np.mean(ref_step**2))),\n                                         induced_normal_shift_rms=float(np.sqrt(np.mean(effects**2))),\n                                         induced_step_rms=float(np.sqrt(np.mean((step - ref_step)**2)))))\n                        if noise == 0.02:\n                            heights[f"{method}_{contrast:g}"] = (values[area] - clean[area]) @ normal\n                if noise == 0.02:\n                    np.savez_compressed(cases / f"{shape}.npz", local=local, **heights)\n            print(f"Paint probe {shape} · {len(area)} planar samples · {len(pairs)} cross-boundary links", flush=True)\n    result = pd.DataFrame(rows)\n    assert result.object_id.nunique() >= 4, "Too few validated flat patches"\n    assert not result.duplicated(["object_id", "xyz_noise", "paint_contrast", "method"]).any()\n    invariant = result[result.method.isin(["frozen_pgd", "geometry_plane"])]\n    assert (invariant.induced_step_rms == 0).all() and (invariant.induced_normal_shift_rms == 0).all()\n    result.to_csv(args.out / "paint_probe_scores.csv", index=False)\n    (args.out / "paint_protocol.json").write_text(json.dumps(dict(plane_radius=RADIUS, clean_height_tolerance=0.002,\n        clean_normal_tolerance_degrees=8, contrasts=CONTRASTS, test_objects=sorted(result.object_id.unique()),\n        baseline="same model and XYZ, uniform grey; treatment paints two grey levels on the clean planar patch"), indent=2))\n    print(f"PAINT PROBE COMPLETE · {result.object_id.nunique()} objects · {len(result)} scores", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "run", "mesh-cache", "plane-model", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    run(parser.parse_args())\n', 'task5/summarise.py': '"""Produce paired object-level contrasts and figures for the Task 5 report."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nKEYS = ["object_id", "xyz_noise"]\nSCALES = {\n    "cd_squared": 1e6,\n    "p2surface_squared": 1e6,\n    "sharp_p2surface_squared": 1e6,\n    "sharp_coverage_squared": 1e6,\n    "texture_p2surface_squared": 1e6,\n    "texture_normal_jump_rms": 1e3,\n}\nCOLOURS = ("original", "constant", "shuffled", "gaussian_0.05", "gaussian_0.15")\nPLOT_COLOURS = {"plain_rgb": "#C24C30", "gated_rgb": "#24658A",\n                "plain_rgb_plane": "#8655A0", "gated_rgb_plane": "#2D8078"}\nLABELS = {"plain_rgb": "Plain RGB residual", "gated_rgb": "Gated RGB residual",\n          "plain_rgb_plane": "Plain RGB plane", "gated_rgb_plane": "Gated RGB plane"}\n\n\ndef bootstrap(values):\n    values = np.asarray(values)\n    rng = np.random.default_rng(1623)\n    if not len(values):\n        return np.nan, np.nan, np.nan\n    means = values[rng.integers(len(values), size=(10000, len(values)))].mean(axis=1)\n    return float(values.mean()), *map(float, np.quantile(means, [0.025, 0.975]))\n\n\ndef paired(frame, method_a, colour_a, method_b, colour_b, metric):\n    a = frame[(frame.method == method_a) & (frame.colour == colour_a)].set_index(KEYS)[metric]\n    b = frame[(frame.method == method_b) & (frame.colour == colour_b)].set_index(KEYS)[metric]\n    assert a.index.is_unique and b.index.is_unique and a.index.sort_values().equals(b.index.sort_values())\n    return pd.concat({"treatment": a, "reference": b}, axis=1).dropna().groupby(level=0).mean()\n\n\ndef record_contrast(frame, label, method_a, colour_a, method_b, colour_b, metric):\n    objects = paired(frame, method_a, colour_a, method_b, colour_b, metric)\n    delta = objects.treatment - objects.reference\n    mu, lo, hi = bootstrap(delta * SCALES[metric])\n    valid = objects.reference.abs() > 1e-12\n    pct, pct_lo, pct_hi = bootstrap(100 * delta[valid] / objects.reference[valid])\n    return dict(contrast=label, metric=metric, treatment=method_a, treatment_colour=colour_a,\n                reference=method_b, reference_colour=colour_b, n_objects=len(delta), wins=int((delta < 0).sum()),\n                equal=int((delta == 0).sum()), mean_treatment=float(objects.treatment.mean() * SCALES[metric]),\n                mean_reference=float(objects.reference.mean() * SCALES[metric]), mean_difference=mu,\n                ci_low=lo, ci_high=hi, object_mean_percent=pct, pct_ci_low=pct_lo, pct_ci_high=pct_hi)\n\n\ndef contrasts(main, planes):\n    requests = []\n    for method in ("plain_rgb", "gated_rgb"):\n        requests.append((main, f"{method}: original vs constant", method, "original", method, "constant"))\n        for colour in ("shuffled", "gaussian_0.05", "gaussian_0.15"):\n            requests.append((main, f"{method}: {colour} vs original", method, colour, method, "original"))\n    for colour in COLOURS:\n        requests.append((main, f"residual gate vs plain: {colour}", "gated_rgb", colour, "plain_rgb", colour))\n        requests.append((planes, f"plane gate vs geometry: {colour}", "gated_rgb_plane", colour, "geometry_plane", colour))\n        requests.append((planes, f"plane gate vs plain: {colour}", "gated_rgb_plane", colour, "plain_rgb_plane", colour))\n    for method in ("geometry_plane", "plain_rgb_plane", "gated_rgb_plane"):\n        requests.append((planes, f"{method}: original vs frozen PGD", method, "original", "frozen_pgd", "original"))\n    return pd.DataFrame([record_contrast(*entry, metric) for entry in requests for metric in SCALES])\n\n\ndef plot_ablation(contrast, out):\n    metrics = [("cd_squared", "Chamfer"), ("p2surface_squared", "Surface error"),\n               ("sharp_coverage_squared", "Sharp edge coverage"), ("texture_p2surface_squared", "Texture surface error")]\n    fig, axes = plt.subplots(2, 2, figsize=(7.6, 3.6), layout="constrained")\n    for axis, (key, title) in zip(axes.flat, metrics, strict=True):\n        rows = contrast[contrast.metric.eq(key) & contrast.contrast.str.endswith("original vs constant")]\n        axis.axvline(0, color="#888888", linewidth=0.8)\n        for y, row in enumerate(rows.itertuples()):\n            lo, hi, mid = row.pct_ci_low, row.pct_ci_high, row.object_mean_percent\n            axis.plot([lo, hi], [y, y], lw=2, color=PLOT_COLOURS[row.treatment])\n            axis.scatter(mid, y, color=PLOT_COLOURS[row.treatment], s=31, zorder=3)\n        axis.set(yticks=[0, 1], yticklabels=["Plain RGB", "Gated RGB"], ylim=(-0.55, 1.55), title=title)\n        axis.set_xlabel("Change from constant RGB (%)", fontsize=8)\n        axis.spines[["top", "right", "left"]].set_visible(False)\n        axis.tick_params(axis="both", labelsize=8)\n    fig.savefig(out / "rgb_ablation.png", dpi=240)\n    plt.close(fig)\n\n\ndef summarise_paint(paint, paint_dir, out):\n    fields = ["induced_normal_shift_rms", "induced_step_rms", "step_rms_change"]\n    per_object = paint.groupby(["method", "paint_contrast", "object_id"])[fields].mean()\n    summary = []\n    for (method, contrast), rows in per_object.groupby(level=[0, 1]):\n        for metric in fields:\n            mean, lo, hi = bootstrap(rows[metric] * 1e3)\n            summary.append(dict(method=method, contrast=contrast, metric=metric, n_objects=len(rows),\n                                mean=mean, ci_low=lo, ci_high=hi,\n                                positive_objects=int((rows[metric] > 0).sum())))\n    result = pd.DataFrame(summary)\n    result.to_csv(out / "paint_summary.csv", index=False)\n    focus = result[result.metric.eq("induced_step_rms")]\n    fig, axis = plt.subplots(figsize=(7.5, 3.0), layout="constrained")\n    for method, label in LABELS.items():\n        rows = focus[focus.method.eq(method)].sort_values("contrast")\n        if rows.empty:\n            continue\n        color = PLOT_COLOURS[method]\n        axis.plot(rows.contrast, rows["mean"], marker="o", color=color, label=label)\n        axis.fill_between(rows.contrast, rows.ci_low, rows.ci_high, color=color, alpha=0.12)\n    axis.set(xlabel="Paint contrast per channel (0–1 RGB scale)",\n             ylabel="Induced normal step RMS × 1,000", xticks=[0, 0.4, 0.8])\n    axis.spines[["top", "right"]].set_visible(False)\n    axis.legend(frameon=False, fontsize=8, ncol=2)\n    fig.savefig(out / "paint_intervention.png", dpi=240)\n    plt.close(fig)\n\n    strongest = paint[(paint.method == "plain_rgb") & (paint.xyz_noise == 0.02) & (paint.paint_contrast == 0.8)]\n    case = strongest.loc[strongest.step_rms_change.idxmax()]\n    with np.load(paint_dir / "paint_cases" / f"{case.object_id}.npz", allow_pickle=False) as saved:\n        local = saved["local"]\n        fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.0), layout="constrained")\n        axes[0].scatter(local[:, 0], local[:, 1], s=10, c=np.where(local[:, 0] >= 0, "#A9B2BE", "#354C66"))\n        axes[0].axvline(0, color="#444444", lw=0.6)\n        axes[0].set(title="Paint applied to a flat patch", xlabel="Across the colour boundary",\n                    ylabel="Along the colour boundary", aspect="equal")\n        for method in PLOT_COLOURS:\n            key = f"{method}_0.8"\n            if key not in saved:\n                continue\n            displacement = (saved[key] - saved[f"{method}_0"]) * 1e3\n            axis_distance = local[:, 0]\n            use = np.abs(local[:, 1]) <= 0.07\n            bins = np.linspace(-0.10, 0.10, 17)\n            centers = 0.5 * (bins[:-1] + bins[1:])\n            values = []\n            for a, b in zip(bins[:-1], bins[1:], strict=True):\n                selected = displacement[use & (axis_distance >= a) & (axis_distance < b)]\n                values.append(selected.mean() if len(selected) >= 3 else np.nan)\n            axes[1].plot(centers, values, marker=".", color=PLOT_COLOURS[method], label=LABELS[method])\n        axes[1].axvline(0, color="#888888", lw=0.6)\n        axes[1].axhline(0, color="#888888", lw=0.6)\n        axes[1].set(title="Output change caused by paint", xlabel="Across the colour boundary",\n                    ylabel="Signed normal displacement × 1,000")\n        axes[1].legend(frameon=False, fontsize=7)\n        for axis in axes:\n            axis.spines[["top", "right"]].set_visible(False)\n            axis.tick_params(labelsize=8)\n        fig.savefig(out / "paint_failure_case.png", dpi=240)\n        plt.close(fig)\n    return result, dict(object_id=str(case.object_id), xyz_noise=float(case.xyz_noise),\n                        contrast=float(case.paint_contrast), plain_step_rms_change=float(case.step_rms_change * 1e3),\n                        n_links=int(case.n_links))\n\n\ndef main(args):\n    args.out.mkdir(parents=True, exist_ok=True)\n    main = pd.read_csv(args.gso / "scores.csv")\n    planes = pd.read_csv(args.planes / "test_scores.csv")\n    paint = pd.read_csv(args.paint / "paint_probe_scores.csv")\n    contrast = contrasts(main, planes)\n    contrast.to_csv(args.out / "paired_contrasts.csv", index=False)\n    for name, frame in (("task4", main), ("plane", planes)):\n        means = frame.groupby(["method", "colour"])[list(SCALES)].mean() * pd.Series(SCALES)\n        means.to_csv(args.out / f"{name}_summary.csv")\n    plot_ablation(contrast, args.out)\n    paint_summary, case = summarise_paint(paint, args.paint, args.out)\n    protocol = dict(bootstrap="10,000 paired resamples of whole test objects after averaging XYZ settings",\n                    failure_case=case,\n                    n_test_objects=int(main.object_id.nunique()),\n                    n_paint_objects=int(paint.object_id.nunique()))\n    (args.out / "summary_metadata.json").write_text(json.dumps(protocol, indent=2))\n    print(json.dumps(protocol), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("gso", "planes", "paint", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    main(parser.parse_args())\n'}
work = Path('/content/mvp_surface_evaluation')
for name, source in sources.items():
    path = work / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)

env = dict(os.environ, OPENBLAS_NUM_THREADS='2', OMP_NUM_THREADS='2',
           MKL_NUM_THREADS='2')
mesh_cache = work / 'meshes'

def execute(script, **flags):
    argv = [sys.executable, '-u', str(work / script)]
    for flag, value in flags.items():
        argv.extend(['--' + flag.replace('_', '-'), str(value)])
    with subprocess.Popen(argv, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
        if proc.wait():
            raise subprocess.CalledProcessError(proc.returncode, argv)


In [ ]:
surface = run / 'surface'
texture = surface / 'texture_regions'
if not (surface / 'per_object_surface.csv').is_file():
    execute('task4/evaluate_surface.py', data=data, run=run, pgd_cache=pgd_cache,
            candidates=candidates, mesh_cache=mesh_cache, out=surface)
if not all((texture / name).is_file() for name in ('clean_point_masks.npz', 'mask_metadata.json')):
    execute('task4/texture_masks.py', data=data, candidates=candidates, out=texture)


In [ ]:
output = PROJECT / 'task5' / run.name
gso = output / 'gso_colour_interventions_v1'
execute('task5/evaluate_gso.py', data=data, run=run, candidates=candidates,
        mesh_cache=mesh_cache, pgd_cache=pgd_cache, out=gso)


In [ ]:
planes = output / 'rgb_neighbourhood_planes_v1'
execute('task5/plane_experts.py', data=data, run=run, candidates=candidates,
        mesh_cache=mesh_cache, out=planes)


In [ ]:
paint = output / 'flat_colour_boundary_v2'
execute('task5/paint_probe.py', data=data, run=run, mesh_cache=mesh_cache,
        plane_model=planes / 'fit.json', out=paint)


In [ ]:
summary = output / 'summary_v1'
execute('task5/summarise.py', gso=gso, planes=planes, paint=paint, out=summary)

import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(summary / 'task4_summary.csv'))
display(pd.read_csv(summary / 'plane_summary.csv'))
for filename in ('rgb_ablation.png', 'paint_intervention.png', 'paint_failure_case.png'):
    display(Image(filename=str(summary / filename)))
